# Ch 2　框架、runtime、harness：LangChain / LangGraph / Deep Agents 的分層地圖

> 同一個需求——「一個會查天氣的 agent」——我們用**三層各寫一次**，直觀感受抽象高度的差異。
>
> **需要**：Google (Gemini) API key（(A) 與 (C) 會真的呼叫模型）。(B) 是不需金鑰、可離線跑的 LangGraph 機制示範。

In [ ]:
!uv pip install -q langchain langgraph deepagents langchain-google-genai grandalf

In [ ]:
import os, getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY：")

# 整章共用的天氣工具。注意 docstring——框架會把它當成「給模型看的工具說明」，所以別省。
def get_weather(city: str) -> str:
    """查詢指定城市的天氣。"""
    return f"{city} 現在晴天，28°C。"

## (A) LangChain：最高抽象、最快上手

Ch1 手刻五十行的 agent loop，這裡 `create_agent` 一行包好。

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[get_weather],
    system_prompt="你是個樂於助人的助理。",
)

result = agent.invoke({"messages": [{"role": "user", "content": "台北天氣如何？"}]})
# 1.0 的標準化內容格式：content_blocks 能裝文字、reasoning、引用等多種區塊。
print(result["messages"][-1].content_blocks)

## (B) LangGraph：最低抽象、最大掌控

為了讓這格「不需金鑰也能跑」，我們用一個**不呼叫 LLM 的玩具圖**來展示「你能親手畫出節點與分支」這件事。真正會呼叫 LLM 的完整圖留到 Ch5。

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    value: int
    log: Annotated[list[str], add]     # reducer：累加，不是覆蓋（Ch6 會細講）

def classify(state): return {"log": [f"分類為 {'big' if state['value'] > 10 else 'small'}"]}
def handle_big(state): return {"log": ["走了 big 這條路"]}
def handle_small(state): return {"log": ["走了 small 這條路"]}
def route(state): return "big" if state["value"] > 10 else "small"   # 你親手定義「往哪走」

b = StateGraph(State)
b.add_node("classify", classify); b.add_node("big", handle_big); b.add_node("small", handle_small)
b.add_edge(START, "classify")
b.add_conditional_edges("classify", route, {"big": "big", "small": "small"})
b.add_edge("big", END); b.add_edge("small", END)
graph = b.compile()

# 程式變多了，但每個節點、每條分支都在你掌握中——這就是下沉到 runtime 換來的東西。
print(graph.invoke({"value": 15, "log": []}))
print(graph.invoke({"value": 3, "log": []}))

### (B) 的拓樸圖：眼見為憑

把剛剛建好的玩具圖拓樸印出來，跟你腦中的形狀對照。

In [ ]:
# draw_ascii() 需要 grandalf（前面 install 已含）；draw_mermaid() 不需要，回傳可貼進任何 mermaid 工具
print("--- ASCII ---")
print(graph.get_graph().draw_ascii())
print("\n--- Mermaid ---")
print(graph.get_graph().draw_mermaid())

## (C) Deep Agents：開箱即用、自帶超能力

寫起來跟 (A) 一樣短，但這個 agent 天生就會規劃、有虛擬檔案系統、能派生 subagents、會自動管理 context。

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    tools=[get_weather],
    system_prompt="你是個樂於助人的研究助理。",
)

# 對「查天氣」這種小事這是殺雞用牛刀；但對「研究一個主題、查 20 個來源、寫成報告」就是天壤之別。
result = agent.invoke({"messages": [{"role": "user", "content": "台北天氣如何？"}]})
print(result["messages"][-1].content_blocks)

## 一句話總結

三者不是競品，是同一堆疊的三層：**LangGraph（runtime）← LangChain（framework）← Deep Agents（harness）**。愈往上內建愈多、你寫得愈少，但黑盒成分也愈高。先學會最底層，上層對你就不再是魔法。